## 1. 데이터 로더
원본 CSV 로딩 + 상수 정의만. 가공은 다음 섹션에서.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

TARGET_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
CAPACITY_KWH = {"kpx_group_1": 21600, "kpx_group_2": 21600, "kpx_group_3": 21000}
ID_COLS = ["forecast_kst_dtm", "data_available_kst_dtm", "grid_id", "latitude", "longitude"]

# 그룹별 확정된 격자 설정 (그룹1만 인접4격자, 실험으로 검증됨 +0.006 / 나머지는 16개 전체평균)
GROUP_LDAPS_GRIDS = {
    "kpx_group_1": [5, 6, 10, 11],
    "kpx_group_2": list(range(1, 17)),
    "kpx_group_3": list(range(1, 17)),
}
GFS_GRID = 5  # 해상도 한계로 3그룹 구분 불가, 공통 사용

GROUP_SCADA_TURBINES = {
    "kpx_group_1": ("vestas", list(range(1, 7))),
    "kpx_group_2": ("vestas", list(range(7, 13))),
    "kpx_group_3": ("unison", list(range(1, 6))),
}


def load_csv(path):
    df = pd.read_csv(path, encoding="utf-8-sig")
    df["forecast_kst_dtm"] = pd.to_datetime(df["forecast_kst_dtm"])
    return df

def load_train_labels():
    df = pd.read_csv(TRAIN_DIR / "train_labels.csv", encoding="utf-8-sig")
    df["kst_dtm"] = pd.to_datetime(df["kst_dtm"])
    return df

def load_scada(maker):
    df = pd.read_csv(TRAIN_DIR / f"scada_{maker}_train.csv", encoding="utf-8-sig")
    df["kst_dtm"] = pd.to_datetime(df["kst_dtm"])
    return df


ldaps_train_raw = load_csv(TRAIN_DIR / "ldaps_train.csv")
gfs_train_raw = load_csv(TRAIN_DIR / "gfs_train.csv")
ldaps_test_raw = load_csv(TEST_DIR / "ldaps_test.csv")
gfs_test_raw = load_csv(TEST_DIR / "gfs_test.csv")
train_labels = load_train_labels()
scada_vestas = load_scada("vestas")
scada_unison = load_scada("unison")

print("ldaps_train:", ldaps_train_raw.shape, "| gfs_train:", gfs_train_raw.shape)
print("ldaps_test:", ldaps_test_raw.shape, "| gfs_test:", gfs_test_raw.shape)
print("train_labels:", train_labels.shape)
print("scada_vestas:", scada_vestas.shape, "| scada_unison:", scada_unison.shape)

ldaps_train: (420864, 35) | gfs_train: (236736, 40)
ldaps_test: (140160, 35) | gfs_test: (78840, 40)
train_labels: (26304, 4)
scada_vestas: (157819, 37) | scada_unison: (105264, 16)


## 2. 전처리
격자집계 + 캘린더 + 데이터셋 조립 + SCADA 풍속보정모델(v4) 학습/적용

- 격자: 그룹1만 인접4격자[5,6,10,11], 나머지는 16개 전체평균 (실험으로 확정)
- 학습기간: 그룹1=2023부터, 그룹2=2022부터, 그룹3=2023부터 (로컬val+리더보드 검증됨)
- v4 보정: SCADA 실측풍속을 타겟으로, 저/고풍속 구간가중치 + 정규화로 과적합 억제

In [2]:
from lightgbm import LGBMRegressor

# 그룹별 확정된 학습 시작점 (로컬val + 리더보드 둘 다로 검증 완료)
TRAIN_PERIOD_START = {
    "kpx_group_1": "2023-01-01",
    "kpx_group_2": "2022-01-01",
    "kpx_group_3": "2023-01-01",
}
CORR_VAL_START = pd.Timestamp("2024-01-01")


def aggregate_grids(df, grids, prefix):
    sub = df[df["grid_id"].isin(grids)]
    value_cols = [c for c in sub.columns if c not in ID_COLS]
    agg = sub.groupby("forecast_kst_dtm")[value_cols].mean()
    agg.columns = [f"{prefix}_{c}" for c in agg.columns]
    return agg.reset_index()


def calendar_features(dt_series):
    dt = pd.to_datetime(dt_series)
    out = pd.DataFrame(index=dt.index)
    out["hour_sin"] = np.sin(2 * np.pi * dt.dt.hour / 24)
    out["hour_cos"] = np.cos(2 * np.pi * dt.dt.hour / 24)
    out["month_sin"] = np.sin(2 * np.pi * dt.dt.month / 12)
    out["month_cos"] = np.cos(2 * np.pi * dt.dt.month / 12)
    return out


def build_dataset(split, group, ldaps, gfs, labels=None):
    ldaps_agg = aggregate_grids(ldaps, GROUP_LDAPS_GRIDS[group], "ldaps")
    gfs_agg = aggregate_grids(gfs, [GFS_GRID], "gfs")
    df = ldaps_agg.merge(gfs_agg, on="forecast_kst_dtm", how="inner")
    df = pd.concat([df, calendar_features(df["forecast_kst_dtm"])], axis=1)

    if labels is not None:
        y = labels[["kst_dtm", group]].rename(columns={"kst_dtm": "forecast_kst_dtm", group: "target"})
        df = df.merge(y, on="forecast_kst_dtm", how="left").dropna(subset=["target"])

    df = df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    feat_cols = [c for c in df.columns if c not in ("forecast_kst_dtm", "target")]
    df[feat_cols] = df[feat_cols].interpolate(method="linear", limit_direction="both")
    return df


def build_scada_wind_target_avg(group):
    maker, turbines = GROUP_SCADA_TURBINES[group]
    src = scada_vestas if maker == "vestas" else scada_unison
    ws_cols = [f"{maker}_wtg{t:02d}_ws" for t in turbines]
    turbine_mean = src[ws_cols].mean(axis=1)
    tmp = pd.DataFrame({"kst_dtm": src["kst_dtm"], "scada_ws": turbine_mean})
    tmp["_hour"] = tmp["kst_dtm"].dt.floor("h")
    hourly = tmp.drop(columns=["kst_dtm"]).groupby("_hour").mean().reset_index()
    return hourly.rename(columns={"_hour": "forecast_kst_dtm"})


def make_group_specific_weight(group, scada_ws):
    weight = np.ones(len(scada_ws))
    if group in ("kpx_group_1", "kpx_group_2"):
        weight = np.where((scada_ws >= 9) & (scada_ws < 12), 5.0, weight)
        weight = np.where(scada_ws >= 12, 3.0, weight)
    else:
        weight = np.where((scada_ws >= 9) & (scada_ws < 12), 2.0, weight)
        weight = np.where((scada_ws >= 12) & (scada_ws < 15), 4.0, weight)
        weight = np.where(scada_ws >= 15, 6.0, weight)
    return weight


# ---- 데이터셋 조립 실행 ----
train_datasets = {g: build_dataset("train", g, ldaps_train_raw, gfs_train_raw, train_labels) for g in TARGET_COLS}
test_datasets = {g: build_dataset("test", g, ldaps_test_raw, gfs_test_raw, None) for g in TARGET_COLS}
RAW_FEATS = [c for c in train_datasets["kpx_group_1"].columns if c not in ("forecast_kst_dtm", "target")]

for g in TARGET_COLS:
    print(f"[{g}] train={train_datasets[g].shape}, test={test_datasets[g].shape}")

# ---- SCADA 풍속보정모델(v4) 학습 + 적용 ----
wind_correction_models_v4 = {}
for g in TARGET_COLS:
    scada_target = build_scada_wind_target_avg(g)
    df_corr = train_datasets[g][["forecast_kst_dtm"] + RAW_FEATS].merge(
        scada_target, on="forecast_kst_dtm", how="inner"
    ).dropna(subset=["scada_ws"])
    tr_corr = df_corr[
        (df_corr["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g])
        & (df_corr["forecast_kst_dtm"] < CORR_VAL_START)
    ]
    sw = make_group_specific_weight(g, tr_corr["scada_ws"].to_numpy())

    model = LGBMRegressor(
        n_estimators=300, max_depth=4, num_leaves=15, min_child_samples=50,
        learning_rate=0.03, reg_alpha=1.0, reg_lambda=1.0, random_state=42, verbose=-1,
    )
    model.fit(tr_corr[RAW_FEATS], tr_corr["scada_ws"], sample_weight=sw)
    wind_correction_models_v4[g] = model
    print(f"[{g}] 풍속보정모델(v4) 학습 완료")

for g in TARGET_COLS:
    train_datasets[g]["corrected_ws_v4"] = wind_correction_models_v4[g].predict(train_datasets[g][RAW_FEATS])
    test_datasets[g]["corrected_ws_v4"] = wind_correction_models_v4[g].predict(test_datasets[g][RAW_FEATS])

FEATS = RAW_FEATS + ["corrected_ws_v4"]
print(f"\n최종 피처 수: {len(FEATS)}")

[kpx_group_1] train=(26200, 71), test=(8760, 70)
[kpx_group_2] train=(26201, 71), test=(8760, 70)
[kpx_group_3] train=(17538, 71), test=(8760, 70)
[kpx_group_1] 풍속보정모델(v4) 학습 완료
[kpx_group_2] 풍속보정모델(v4) 학습 완료
[kpx_group_3] 풍속보정모델(v4) 학습 완료

최종 피처 수: 70


## 3. 모델 정의
평가지표(대회 공식 metric) + Quantile Regression 학습/예측 함수 + 확정된 alpha/seed

- 일반회귀(평균예측) 대신 Quantile Regression 사용 → 저/고풍속 극단 구간의 편향 완화
- alpha는 세밀스캔으로 그룹별 확정 (0.5=중앙값, 클수록 과대예측 방향)
- seed 3개 앙상블로 학습 노이즈 상쇄

In [3]:
BEST_ALPHA = {
    "kpx_group_1": 0.75,
    "kpx_group_2": 0.60,
    "kpx_group_3": 0.75,
}
SEEDS = [42, 1, 2]


def group_score(actual, forecast, capacity):
    valid = actual >= capacity * 0.10
    actual_v, forecast_v = actual[valid], forecast[valid]
    error_rate = np.abs(forecast_v - actual_v) / capacity
    nmae = np.mean(error_rate)
    unit_price = np.select([error_rate <= 0.06, error_rate <= 0.08], [4.0, 3.0], default=0.0)
    earned = np.sum(actual_v * unit_price)
    maxset = np.sum(actual_v * 4.0)
    ficr = earned / maxset
    return 0.5 * (1 - nmae) + 0.5 * ficr, nmae, ficr


def make_quantile_model(alpha, seed):
    return LGBMRegressor(
        objective="quantile", alpha=alpha, n_estimators=300,
        max_depth=8, learning_rate=0.05, random_state=seed, verbose=-1,
    )


def fit_predict_ensemble(X_train, y_train, X_predict, alpha, seeds, capacity):
    preds = []
    for seed in seeds:
        model = make_quantile_model(alpha, seed)
        model.fit(X_train, y_train)
        preds.append(model.predict(X_predict))
    ensemble_pred = np.mean(preds, axis=0)
    return np.clip(ensemble_pred, 0, capacity)

print("모델 정의 완료. BEST_ALPHA:", BEST_ALPHA, "| SEEDS:", SEEDS)

모델 정의 완료. BEST_ALPHA: {'kpx_group_1': 0.75, 'kpx_group_2': 0.6, 'kpx_group_3': 0.75} | SEEDS: [42, 1, 2]


## 4. 추론
(a) val(2024) 예측 - 성능 확인용, leakage 없이 그룹별 시작점~2023만 학습
(b) test(2025) 예측 - 제출용, 그룹별 시작점~2024 전체로 재학습

In [4]:
# ---- (a) val(2024) 예측 ----
val_results = {}
for g in TARGET_COLS:
    df = train_datasets[g]
    tr = df[(df["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g]) & (df["forecast_kst_dtm"] < CORR_VAL_START)]
    va = df[df["forecast_kst_dtm"] >= CORR_VAL_START].copy()
    capacity = CAPACITY_KWH[g]

    va["pred"] = fit_predict_ensemble(
        tr[FEATS], tr["target"], va[FEATS], BEST_ALPHA[g], SEEDS, capacity
    )
    val_results[g] = va
    print(f"[{g}] val 예측 완료 (n={len(va)})")


# ---- (b) test(2025) 예측 ----
test_predictions = {}
for g in TARGET_COLS:
    df = train_datasets[g]
    tr_full = df[df["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g]]
    capacity = CAPACITY_KWH[g]

    pred = fit_predict_ensemble(
        tr_full[FEATS], tr_full["target"], test_datasets[g][FEATS], BEST_ALPHA[g], SEEDS, capacity
    )
    test_predictions[g] = pd.Series(pred, index=test_datasets[g]["forecast_kst_dtm"].values)
    print(f"[{g}] test(2025) 최종학습+예측 완료 (n_train={len(tr_full)})")

[kpx_group_1] val 예측 완료 (n=8779)
[kpx_group_2] val 예측 완료 (n=8779)
[kpx_group_3] val 예측 완료 (n=8779)
[kpx_group_1] test(2025) 최종학습+예측 완료 (n_train=17536)
[kpx_group_2] test(2025) 최종학습+예측 완료 (n_train=26201)
[kpx_group_3] test(2025) 최종학습+예측 완료 (n_train=17538)


## 5. 결과
val 성능 요약 + 구간별 오차율 진단 + submission.csv 생성

- 구간별(target_bin) 진단: 발전량 10~70% 구간이 여전히 8% 경계를 넘는 경향 있음 (다음 개선 포인트)

In [5]:
# ---- val(2024) 성능 요약 ----
print("=" * 60)
print("VAL(2024) 성능 요약")
print("=" * 60)
scores = []
for g in TARGET_COLS:
    va = val_results[g]
    capacity = CAPACITY_KWH[g]
    y_va = va["target"].to_numpy(dtype=float)
    pred = va["pred"].to_numpy(dtype=float)
    score, nmae, ficr = group_score(y_va, pred, capacity)
    scores.append(score)
    print(f"[{g}] score={score:.4f}  nmae={nmae:.4f}  ficr={ficr:.4f}")
print(f"\n전체 평균 score: {np.mean(scores):.4f}")


# ---- 구간별(target_bin) 오차율 진단 ----
print("\n" + "=" * 60)
print("구간별(target_bin) 오차율 진단")
print("=" * 60)
for g in TARGET_COLS:
    va = val_results[g].copy()
    capacity = CAPACITY_KWH[g]
    va["error_rate"] = np.abs(va["pred"] - va["target"]) / capacity
    va["target_pct"] = va["target"] / capacity
    va["target_bin"] = pd.cut(
        va["target_pct"], bins=[0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.01],
        labels=["0-10%", "10-30%", "30-50%", "50-70%", "70-90%", "90-100%"],
    )
    print(f"\n[{g}]")
    print(va.groupby("target_bin", observed=True)["error_rate"].mean().round(4))


# ---- submission.csv 생성 ----
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv", encoding="utf-8-sig")
sample_submission["forecast_kst_dtm"] = pd.to_datetime(sample_submission["forecast_kst_dtm"])

submission = sample_submission[["forecast_id", "forecast_kst_dtm"]].copy()
for g in TARGET_COLS:
    submission[g] = submission["forecast_kst_dtm"].map(test_predictions[g])

submission["forecast_kst_dtm"] = submission["forecast_kst_dtm"].dt.strftime("%Y-%m-%d %H:%M:%S")
output_path = DATA_DIR.parent / "submission.csv"
submission.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"\nSaved: {output_path}")
print(f"shape={submission.shape}  NaN={submission[TARGET_COLS].isna().sum().sum()}")
submission.head()

VAL(2024) 성능 요약
[kpx_group_1] score=0.6466  nmae=0.1332  ficr=0.4263
[kpx_group_2] score=0.6646  nmae=0.1272  ficr=0.4564
[kpx_group_3] score=0.5863  nmae=0.1487  ficr=0.3214

전체 평균 score: 0.6325

구간별(target_bin) 오차율 진단

[kpx_group_1]
target_bin
0-10%      0.0872
10-30%     0.1476
30-50%     0.1750
50-70%     0.1286
70-90%     0.0959
90-100%    0.0777
Name: error_rate, dtype: float64

[kpx_group_2]
target_bin
0-10%      0.0622
10-30%     0.1406
30-50%     0.1594
50-70%     0.1380
70-90%     0.0921
90-100%    0.0746
Name: error_rate, dtype: float64

[kpx_group_3]
target_bin
0-10%      0.0746
10-30%     0.1536
30-50%     0.1646
50-70%     0.1457
70-90%     0.1128
90-100%    0.1631
Name: error_rate, dtype: float64

Saved: submission.csv
shape=(8760, 5)  NaN=0


,forecast_id,forecast_kst_dtm,kpx_group_1,kpx_group_2,kpx_group_3
0,forecast_0001,2025-01-01 01:00:00,20733.405627,19138.114580,19210.519486
1,forecast_0002,2025-01-01 02:00:00,20322.596785,19083.835440,19094.864961
2,forecast_0003,2025-01-01 03:00:00,19815.501531,18688.331813,17077.003793
3,forecast_0004,2025-01-01 04:00:00,18407.110451,18161.645938,15776.973419
4,forecast_0005,2025-01-01 05:00:00,18056.753616,16698.633480,15542.930329
